# Stage07B — Colab A100 anchor only
This notebook is only the VS Code ↔ Colab runtime anchor. Heavy commands are intended to run in the **Colab terminal** using ordinary shell commands (`cd`, `python`, `pip`), not notebook magics.


In [ ]:
print("Hello, World!")

Hello, World!


In [2]:
import torch, os
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0)
    print('GPU:', p.name)
    print('VRAM GiB:', p.total_memory/2**30)
print('Notebook PID:', os.getpid())


CUDA: True
GPU: NVIDIA L4
VRAM GiB: 22.0343017578125
Notebook PID: 12776


In [3]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive/DSC2026/stage07b')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent output root:', DRIVE_ROOT)


Mounted at /content/drive
Persistent output root: /content/drive/MyDrive/DSC2026/stage07b


## Terminal workflow
After uploading only `stage07b_colab_payload.tar.gz` to `/content`, use the Colab terminal:
```bash
cp "/content/drive/MyDrive/DSC2026/stage07b/stage07b_colab_payload.tar.gz" /content/
tar -xzf stage07b_colab_payload.tar.gz
pip install -q 'transformers>=4.51.0' accelerate safetensors scikit-learn
unzip -o "/content/drive/MyDrive/DSC2026/stage07b/stage07n_l4_bf16_fix.zip" -d /content
python /content/stage07b/run_qwen06b_gold_supervised_l4.py

cp "/content/drive/MyDrive/DSC2026/stage07b/stage07b_student_resume_v5.zip" /content/
unzip -o /content/stage07b_student_resume_v5.zip \
  -d /content/stage07b

export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
python /content/stage07b/run_qwen06b_student_portable_a100_v5.py \
  --drive-root /content/drive/MyDrive/DSC2026/stage07b \
  --smoke-test
python /content/stage07b/run_qwen06b_student_portable_a100_v5.py \
  --drive-root /content/drive/MyDrive/DSC2026/stage07b \
  --folds 0

chmod +x /content/stage07b/run_stage07b_student_resume_v5.sh
bash /content/stage07b/run_stage07b_student_resume_v5.sh

chmod +x /content/stage07b/run_stage07b_unattended.sh
bash /content/stage07b/run_stage07b_unattended.sh

unzip -o "/content/drive/MyDrive/DSC2026/stage07b/stage07n_qwen06b_gold_supervised.zip" -d /content
python /content/stage07b/run_qwen06b_gold_supervised_dev_cert.py --eval-batch-queries 2
```
The unattended wrapper checkpoints to Drive, logs to Drive, and unassigns the runtime when it finishes **or when teacher/student exits with an error**.


## Manual emergency shutdown
**Run the next cell only when you are ready to terminate the runtime.** It flushes Drive first, then unassigns/deletes the Colab runtime.


In [ ]:
from google.colab import drive, runtime
try:
    drive.flush_and_unmount()
except Exception as e:
    print('Drive flush warning:', e)
runtime.unassign()
